In [ ]:
import pandas as pd
import joblib

model = joblib.load("../../models/random_forest_model.pkl")

In [ ]:
future_matches = pd.read_csv('../../data/pl25-26.csv')

In [ ]:
future_matches = future_matches[[
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC",
    "HY", "AY", "HR", "AR",
    "B365H", "B365D", "B365A"
]]

In [ ]:
rename_map = {
    "Date": "date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR":  "result",

    "HTHG": "home_ht_goals",
    "HTAG": "away_ht_goals",
    "HTR":  "ht_result",

    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_sot",
    "AST": "away_sot",
    "HF": "home_fouls",
    "AF": "away_fouls",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow",
    "AY": "away_yellow",
    "HR": "home_red",
    "AR": "away_red",

    "B365H": "odds_home_win",
    "B365D": "odds_draw",
    "B365A": "odds_away_win"
}
future_matches.rename(columns=rename_map, inplace=True)

In [ ]:
future_matchesXG = pd.read_csv('../../data/pl_all_seasonsXG.csv')

In [ ]:
team_name_map = {
        "Newcastle Utd": "Newcastle",
        "Manchester Utd": "Man United",
        "Manchester City": "Man City",
        "Leicester City": "Leicester",
        "Leeds United": "Leeds",
        "Ipswich Town": "Ipswich",
        "Luton Town": "Luton",
        "Norwich City": "Norwich",
        "Sheffield Utd": "Sheffield United",
        "Nott'ham Forest": "Nott'm Forest"
    }

future_matchesXG['home_team'] = future_matchesXG['home_team'].replace(team_name_map)
future_matchesXG['away_team'] = future_matchesXG['away_team'].replace(team_name_map)

In [ ]:
future_matches['date'] = pd.to_datetime(future_matches['date'])
future_matchesXG['date'] = pd.to_datetime(future_matchesXG['date'])

In [ ]:
merged = future_matches.merge(future_matchesXG, on=['date', 'home_team', 'away_team'], how='left')

In [ ]:
merged.info()

In [ ]:
import pandas as pd

def add_rolling_team_stats(df, window=5):

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    stat_map = {
        "goals_scored": ("home_goals", "away_goals"),
        "goals_conceded": ("away_goals", "home_goals"),
        "shots": ("home_shots", "away_shots"),
        "shots_on_target": ("home_sot", "away_sot"),
        "fouls": ("home_fouls", "away_fouls"),
        "corners": ("home_corners", "away_corners"),
        "yellow_cards": ("home_yellow", "away_yellow"),
        "red_cards": ("home_red", "away_red"),
        "xG": ("home_xG", "away_xG"),
        "xG.1": ("away_xG", "home_xG"),
    }

    # Build long format
    home_df = df.rename(columns={v[0]: k for k, v in stat_map.items()})
    away_df = df.rename(columns={v[1]: k for k, v in stat_map.items()})

    home_df["team"] = home_df["home_team"]
    away_df["team"] = away_df["away_team"]

    cols = ["date", "team"] + list(stat_map.keys())
    long_df = pd.concat([home_df[cols], away_df[cols]], ignore_index=True)
    long_df = long_df.sort_values(["team", "date"]).reset_index(drop=True)

    # Compute rolling stats (fixed)
    for stat in stat_map.keys():
        long_df[f"{stat}_rolling_avg"] = (
            long_df.groupby("team")[stat]
                   .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )

    # HOME merge
    home_merge_cols = {f"{stat}_rolling_avg": f"home_{stat}_rolling"
                       for stat in stat_map.keys()}

    df = df.merge(
        long_df[["date", "team"] + list(home_merge_cols.keys())],
        left_on=["date", "home_team"],
        right_on=["date", "team"],
        how="left"
    ).drop(columns=["team"])
    df = df.rename(columns=home_merge_cols)

    # AWAY merge
    away_merge_cols = {f"{stat}_rolling_avg": f"away_{stat}_rolling"
                       for stat in stat_map.keys()}

    df = df.merge(
        long_df[["date", "team"] + list(away_merge_cols.keys())],
        left_on=["date", "away_team"],
        right_on=["date", "team"],
        how="left"
    ).drop(columns=["team"])
    df = df.rename(columns=away_merge_cols)

    # Fill NAN
    df = df.fillna(0)

    return df


In [ ]:
match_stats = add_rolling_team_stats(merged, window=5)

In [ ]:
match_stats.info()

In [ ]:
import numpy as np
import pandas as pd

def expected_result(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, score_a, score_b, k=20):
    expected_a = expected_result(elo_a, elo_b)
    expected_b = 1 - expected_a

    if score_a > score_b:   # home win
        actual_a, actual_b = 1, 0
    elif score_a < score_b: # away win
        actual_a, actual_b = 0, 1
    else:
        actual_a, actual_b = 0.5, 0.5

    new_a = elo_a + k * (actual_a - expected_a)
    new_b = elo_b + k * (actual_b - expected_b)

    return new_a, new_b

def add_elo_features(df):
    df = df.sort_values("date").copy()

    team_elos = {}
    base_elo = 1500

    df["home_elo_before"] = 0.0
    df["away_elo_before"] = 0.0

    for i, row in df.iterrows():
        home, away = row["home_team"], row["away_team"]

        # If new team, initialize
        team_elos.setdefault(home, base_elo)
        team_elos.setdefault(away, base_elo)

        # Assign ELO before match
        df.at[i, "home_elo_before"] = team_elos[home]
        df.at[i, "away_elo_before"] = team_elos[away]

        # Update after match
        new_home, new_away = update_elo(
            team_elos[home], team_elos[away],
            row["home_goals"], row["away_goals"]
        )

        team_elos[home] = new_home
        team_elos[away] = new_away

    return df


In [ ]:
match_stats = add_elo_features(match_stats)

In [ ]:
def add_bookmaker_features(df):
    df = df.copy()

    # Convert to implied probabilities
    df["prob_home"] = 1 / df["odds_home_win"]
    df["prob_draw"] = 1 / df["odds_draw"]
    df["prob_away"] = 1 / df["odds_away_win"]

    # Normalize to remove overround
    total = df[["prob_home", "prob_draw", "prob_away"]].sum(axis=1)

    df["prob_home"] /= total
    df["prob_draw"] /= total
    df["prob_away"] /= total

    return df

In [ ]:
match_stats = add_bookmaker_features(match_stats)

In [ ]:
match_stats = match_stats.sort_values('date')

match_stats['home_xG_rolling'] = (
    match_stats.groupby('home_team')['xG'].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
)

match_stats['away_xG_rolling'] = (
    match_stats.groupby('away_team')['xG.1'].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
)

match_stats['xG_diff_rolling'] = match_stats['home_xG_rolling'] - match_stats['away_xG_rolling']

match_stats['elo_diff'] = match_stats['home_elo_before'] - match_stats['away_elo_before']

match_stats['goals_diff_rolling'] = match_stats['home_goals_scored_rolling'] - match_stats['away_goals_scored_rolling']
match_stats['conceded_diff_rolling'] = match_stats['home_goals_conceded_rolling'] - match_stats['away_goals_conceded_rolling']

In [ ]:
predict_matches = match_stats[['date', 'home_team', 'away_team', 'odds_home_win', 'odds_draw', 'odds_away_win',
       'home_goals_scored_rolling', 'home_goals_conceded_rolling',
       'home_shots_rolling', 'home_shots_on_target_rolling',
       'home_fouls_rolling', 'home_corners_rolling',
       'home_yellow_cards_rolling', 'away_goals_scored_rolling',
       'away_goals_conceded_rolling', 'away_shots_rolling',
       'away_shots_on_target_rolling', 'away_fouls_rolling',
       'away_corners_rolling', 'away_yellow_cards_rolling', 'home_elo_before',
       'away_elo_before', 'prob_home', 'prob_draw', 'prob_away',
       'home_xG_rolling', 'away_xG_rolling', 'xG_diff_rolling', 'elo_diff',
       'goals_diff_rolling', 'conceded_diff_rolling', 'result']]

In [ ]:
predict_matches.info()

In [ ]:
predict_matches.head(10)

In [ ]:
def predict_match(model, match_row):
    """
    match_row = a single-row DataFrame with the same columns as X
    """
    home = match_row['home_team'].values[0]
    away = match_row['away_team'].values[0]
    probs = model.predict_proba(match_row.drop(['home_team', 'away_team', 'result', 'date'], axis=1))[0]

    print(f"Predicting match: {home} vs {away}")
    print(f'Probabilities:')
    print(f'  Home win: {probs[2]:.2%}')
    print(f'  Draw:     {probs[0]:.2%}')    
    print(f'  Away win: {probs[1]:.2%}')
    return probs


In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Sunderland') & (predict_matches['away_team'] == 'Bournemouth')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Brentford') & (predict_matches['away_team'] == 'Burnley')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Man City') & (predict_matches['away_team'] == 'Leeds')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Everton') & (predict_matches['away_team'] == 'Newcastle')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Tottenham') & (predict_matches['away_team'] == 'Fulham')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == 'Crystal Palace') & (predict_matches['away_team'] == 'Man United')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == "Nott'm Forest") & (predict_matches['away_team'] == 'Brighton')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == "Aston Villa") & (predict_matches['away_team'] == 'Wolves')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == "West Ham") & (predict_matches['away_team'] == 'Liverpool')])
print(probas)

In [ ]:
probas = predict_match(model, predict_matches[(predict_matches['home_team'] == "Chelsea") & (predict_matches['away_team'] == 'Arsenal')])
print(probas)